In [68]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
import os

In [69]:
documents = [
    # Health documents
    Document(page_content="Drinking at least 8 glasses of water a day keeps your body hydrated and supports organ function"),
    Document(page_content="Regular exercise such as 30 minutes of walking daily reduces the risk of heart disease and improves mood"),
    Document(page_content="Eating a balanced diet rich in fruits, vegetables, whole grains, and lean proteins promotes long-term health"),
    Document(page_content="Getting 7 to 9 hours of quality sleep each night is essential for mental clarity and physical recovery"),
    Document(page_content="Managing stress through meditation, deep breathing, or yoga significantly improves overall well-being"),

    # Irrelevant documents
    Document(page_content="The Eiffel Tower in Paris was constructed in 1889 and stands 330 metres tall"),
    Document(page_content="Python is a high-level programming language known for its simple syntax and wide use in data science"),
    Document(page_content="The stock market experienced significant volatility during the 2008 financial crisis"),
    Document(page_content="Jupiter is the largest planet in the solar system and has at least 95 known moons"),
    Document(page_content="The FIFA World Cup is held every four years and is the most watched sporting event on the planet"),
]

In [70]:
embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6565.95it/s]


In [71]:
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embedding_model
)

In [72]:
similarity_retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs={"k" : 6}
)

In [73]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
    max_new_tokens=256,
    temperature=0.5
)

chat_model = ChatHuggingFace(llm=llm)

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k": 6}),
    llm=chat_model
)

In [74]:
query = "How to stay healthy?"

In [75]:
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [76]:
for i, doc in enumerate(similarity_results):
    print(f"\n ---Result Similarity {i+1}---")
    print(doc.page_content)


print("*" * 150)

for i, doc in enumerate(multiquery_results):
    print(f"\n ---multiquery_results {i+1}---")
    print(doc.page_content)



 ---Result Similarity 1---
Eating a balanced diet rich in fruits, vegetables, whole grains, and lean proteins promotes long-term health

 ---Result Similarity 2---
Managing stress through meditation, deep breathing, or yoga significantly improves overall well-being

 ---Result Similarity 3---
Regular exercise such as 30 minutes of walking daily reduces the risk of heart disease and improves mood

 ---Result Similarity 4---
Drinking at least 8 glasses of water a day keeps your body hydrated and supports organ function

 ---Result Similarity 5---
Getting 7 to 9 hours of quality sleep each night is essential for mental clarity and physical recovery

 ---Result Similarity 6---
The FIFA World Cup is held every four years and is the most watched sporting event on the planet
******************************************************************************************************************************************************

 ---multiquery_results 1---
Eating a balanced diet rich in fruits, v